# Algoritmo de Colonia de Hormigas
Daniela Lizet Cauich Lopez  
Código de: Benítez Iglésias, R. (2014). *Inteligencia artificial avanzada*. Editorial UOC.

# Problema del Viajero

## 1. Descripción general
El Problema del Viajero (TSP, por sus siglas en inglés) es un problema clásico de optimización combinatoria que consiste en encontrar la ruta más corta posible que permita a un viajero visitar un conjunto de ciudades exactamente una vez y regresar al punto de origen.

Formalmente, se modela como un grafo completo ponderado, donde:
- Los nodos representan ciudades.
- Las aristas representan las distancias o costos entre ciudades.

El objetivo es determinar el ciclo Hamiltoniano de costo mínimo. Este problema pertenece a la clase de complejidad NP-hard, lo que implica que no existe un algoritmo conocido que lo resuelva de manera eficiente (en tiempo polinomial) para todos los casos.

---

## 2. Objetivos, limitaciones y trabajos previos

### 2.1 Objetivos
- Minimizar la distancia total (o costo) del recorrido.
- Encontrar una solución óptima global.
- Diseñar algoritmos eficientes que reduzcan el tiempo de cómputo en instancias grandes.

---

### 2.2 Limitaciones
- **Complejidad computacional**: El número de posibles rutas crece factorialmente (n!), lo que hace inviable la búsqueda exhaustiva para grandes conjuntos de ciudades.
- **Escalabilidad**: A medida que aumenta el número de nodos, los métodos exactos se vuelven imprácticos.
- **Dependencia del modelo**: Variantes del problema (simétrico, asimétrico, con restricciones adicionales) incrementan la dificultad.
- **Óptimos locales**: Muchos métodos heurísticos pueden quedar atrapados en soluciones subóptimas.

---

### 2.3 Trabajos previos
El TSP ha sido ampliamente estudiado, dando lugar a múltiples enfoques:

#### Métodos exactos
- **Fuerza bruta**: Evalúa todas las posibles permutaciones.
- **Programación dinámica (Held-Karp)**: Reduce la complejidad a O(n²·2ⁿ).
- **Branch and Bound**: Reduce el espacio de búsqueda mediante poda de soluciones no prometedoras.

#### Métodos heurísticos
- **Vecino más cercano (Nearest Neighbor)**.
- **Inserción (más cercana, más lejana, etc.)**.
- **2-opt y 3-opt**: Mejoras locales sobre rutas existentes.

#### Metaheurísticas
- **Algoritmos genéticos**.
- **Enjambre de partículas (PSO)**.
- **Colonia de hormigas (ACO)**.
- **Recocido simulado (Simulated Annealing)**.

Estos enfoques buscan soluciones aproximadas en tiempos razonables, siendo especialmente útiles en problemas de gran escala.

---

## Conclusión
El Problema del Viajero representa un desafío fundamental en optimización debido a su alta complejidad. Su estudio ha impulsado el desarrollo de técnicas tanto exactas como aproximadas, con aplicaciones en logística, planificación de rutas, manufactura y redes.

---
## 2. Algoritmo de Colonia de Hormigas (ACO)

### 2.1 Principio de funcionamiento

Las hormigas reales depositan **feromonas** en su camino. Otros individuos prefieren seguir caminos con más feromonas, y como las rutas más cortas se recorren más rápido, acumulan más feromonas. Con el tiempo, la colonia converge hacia la ruta óptima.

**Parámetros clave:**
- `alfa` ($\alpha = 1.0$): Peso de las feromonas en la decisión
- `beta` ($\beta = 0.5$): Peso de la distancia (heurística)
- `evaporación` ($\rho = 0.1$, es decir, se multiplica por 0.9): Evita que las feromonas antiguas dominen indefinidamente

La probabilidad de que una hormiga en la ciudad $i$ elija la ciudad $j$ es:

$$P_{ij} = \frac{\tau_{ij}^{\alpha} \cdot \eta_{ij}^{\beta}}{\sum_{k \in \text{disponibles}} \tau_{ik}^{\alpha} \cdot \eta_{ik}^{\beta}}$$

donde $\tau_{ij}$ es la feromona en el arco $(i,j)$ y $\eta_{ij} = 1/d_{ij}$ es la visibilidad (inversa de la distancia).

## 3. Código fuente en Python
Fuente: Benítez Iglésias (2014) (pp. 237–240)

In [2]:
import random
import sys
import math

# Nota: en lugar de matrices se usan listas de listas

# Genera una matriz de distancias de nCiudades x nCiudades
def matrizDistancias(nCiud, distanciaMaxima):
    matriz = [[0 for i in range(nCiud)] for j in range(nCiud)]

    for i in range(nCiud):
        for j in range(i):
            matriz[i][j] = distanciaMaxima * random.random()
            matriz[j][i] = matriz[i][j]

    return matriz


# Elige un paso de una hormiga, teniendo en cuenta las distancias
# y las feromonas y descartando las ciudades ya visitadas.
def eligeCiudad(dists, ferom, visitadas):
    # Se calcula la tabla de pesos de cada ciudad
    listaPesos  = []
    disponibles = []
    actual = visitadas[-1]

    # Influencia de cada valor (alfa: feromonas; beta: distancias)
    alfa = 1.0
    beta = 0.5

    # El parámetro beta (peso de las distancias) es 0.5, alfa=1.0
    for i in range(len(dists)):
        if i not in visitadas:
            fer   = math.pow((1.0 + ferom[actual][i]), alfa)
            peso  = math.pow(1.0 / dists[actual][i], beta) * fer
            disponibles.append(i)
            listaPesos.append(peso)

    # Se elige aleatoriamente una de las ciudades disponibles,
    # teniendo en cuenta su peso relativo.
    valor      = random.random() * sum(listaPesos)
    acumulado  = 0.0
    i          = -1
    while valor > acumulado:
        i         += 1
        acumulado += listaPesos[i]

    return disponibles[i]


# Genera una "hormiga", que elegirá un camino teniendo en cuenta
# las distancias y los rastros de feromonas. Devuelve una tupla
# con el camino y su longitud.
def eligeCamino(distancias, feromonas):
    # La ciudad inicial siempre es la 0
    camino      = [0]
    longCamino  = 0

    # Elegir cada paso según la distancia y las feromonas
    while len(camino) < len(distancias):
        ciudad      = eligeCiudad(distancias, feromonas, camino)
        longCamino += distancias[camino[-1]][ciudad]
        camino.append(ciudad)

    # Para terminar hay que volver a la ciudad de origen (0)
    longCamino += distancias[camino[-1]][0]
    camino.append(0)

    return (camino, longCamino)


# Actualiza la matriz de feromonas siguiendo el camino recibido
def rastroFeromonas(feromonas, camino, dosis):
    for i in range(len(camino) - 1):
        feromonas[camino[i]][camino[i + 1]] += dosis


# Evapora todas las feromonas multiplicándolas por una constante
# = 0.9 (en otras palabras, el coeficiente de evaporación es 0.1)
def evaporaFeromonas(feromonas):
    for lista in feromonas:
        for i in range(len(lista)):
            lista[i] *= 0.9


# Resuelve el problema del viajante de comercio mediante el
# algoritmo de la colonia de hormigas. Recibe una matriz de
# distancias y devuelve una tupla con el mejor camino que ha
# obtenido (lista de índices) y su longitud
def hormigas(distancias, iteraciones, distMedia):
    # Primero se crea una matriz de feromonas vacía
    n          = len(distancias)
    feromonas  = [[0 for i in range(n)] for j in range(n)]

    # El mejor camino y su longitud (inicialmente "infinita")
    mejorCamino     = []
    longMejorCamino = sys.maxsize

    # En cada iteración se genera una hormiga, que elige un camino,
    # y si es mejor que el mejor que teníamos, deja su rastro de
    # feromonas (mayor cuanto más corto sea el camino)
    for iter in range(iteraciones):
        (camino, longCamino) = eligeCamino(distancias, feromonas)

        if longCamino <= longMejorCamino:
            mejorCamino      = camino
            longMejorCamino  = longCamino

        rastroFeromonas(feromonas, camino, distMedia / longCamino)

        # En cualquier caso, las feromonas se van evaporando
        evaporaFeromonas(feromonas)

    # Se devuelve el mejor camino que se haya encontrado
    return (mejorCamino, longMejorCamino)


# Generación de una matriz de prueba
numCiudades    = 10
distanciaMaxima = 10
ciudades        = matrizDistancias(numCiudades, distanciaMaxima)

# Obtención del mejor camino
iteraciones = 1000
distMedia   = numCiudades * distanciaMaxima / 2
(camino, longCamino) = hormigas(ciudades, iteraciones, distMedia)
print("Camino: ", camino)
print("Longitud del camino: ", longCamino)

Camino:  [0, 9, 4, 2, 7, 1, 6, 5, 3, 8, 0]
Longitud del camino:  19.950525450271243
